In [8]:
reset()

_.<x> = QQ[]
F = QQ.extension(x^3-x^2-x-1, 'a').galois_closure('b')
l = (x^3-x^2-x-1).roots(ring=F, multiplicities=False)
dl = l[2]/l[0]-l[1]/l[0]-l[2]/l[1]+l[0]/l[1]+l[1]/l[2]-l[0]/l[2]
c = [(1/l[1]-1/l[2])/dl, (1/l[2]-1/l[0])/dl, (1/l[0]-1/l[1])/dl]

def trib(n):
  return QQ(c[0]*l[0]^n + c[1]*l[1]^n + c[2]*l[2]^n)

def N_bound(p):
  _.<T> = GF(p)[]
  n_roots = len((T^3-T^2-T-1).roots(ring=GF(p), multiplicities=False))
  if n_roots == 3:
    return p-1
  elif n_roots == 1:
    return p^2-1
  elif n_roots == 0:
    return p^2+p+1
  else:
    raise ValueError(f"Unexpected number of roots for p = {p}: {n_roots}")

@parallel(max(os.cpu_count()-2,1))  # leave 2 cores free for other tasks
def N0(p):
  # print(p, end=' ')
  for n in sorted(divisors(N_bound(p))):
    if trib(n) % p == 0 and trib(n-1) % p == 0:
      p21 = 1 if trib(n-1) % p^2 == 0 else 0
      p22 = 1 if trib(n) % p^2 == 0 else 0
      if p21 or p22:
        print(f"p = {p}, n = {n}, {p21}{p22}")
      return (n,str(p21)+str(p22))
  else:
    raise ValueError(f"Could not find n such that trib(n) and trib(n-1) are divisible by p = {p}")

[(p,trib(N_bound(p)) % p) for p in primes(start=13,stop=100)]
# [(p[0][0],q) for p,q in N0([p for p in primes(start=13,stop=10000)])]

[(13, 0),
 (17, 0),
 (19, 0),
 (23, 0),
 (29, 0),
 (31, 0),
 (37, 0),
 (41, 0),
 (43, 0),
 (47, 0),
 (53, 0),
 (59, 0),
 (61, 0),
 (67, 0),
 (71, 0),
 (73, 0),
 (79, 0),
 (83, 0),
 (89, 0),
 (97, 0)]

In [7]:
reset()

_.<x> = QQ[]
# F = QQ.extension(x^3-x^2-x-1, 'a').galois_closure('b')
F.<b> = (x^3-x^2-x-1).splitting_field(simplify_all=True)
R = F.ring_of_integers()
l = (x^3-x^2-x-1).roots(ring=R, multiplicities=False)
dl = l[2]/l[0]-l[1]/l[0]-l[2]/l[1]+l[0]/l[1]+l[1]/l[2]-l[0]/l[2]
c = [(1/l[1]-1/l[2])/dl, (1/l[2]-1/l[0])/dl, (1/l[0]-1/l[1])/dl]

def power_mod(base, exp, mod):
  I = mod*R  # ideal generated by mod
  result = R(1)
  base = I.reduce(R(base))
  while exp > 0:
    if exp % 2 == 1:  # If exp is odd
      result = I.reduce(result * base)
    base = I.reduce(base * base)
    exp //= 2
  return result

def trib(n, m): # Tribonacci number T_n mod m
  return QQ(c[0]*power_mod(l[0],n,m) + c[1]*power_mod(l[1],n,m) + c[2]*power_mod(l[2],n,m)) % m

def N_bound(p):
  _.<T> = GF(p)[]
  n_roots = len((T^3-T^2-T-1).roots(ring=GF(p), multiplicities=False))
  if n_roots == 3:
    return p-1
  elif n_roots == 1:
    return p^2-1
  elif n_roots == 0:
    return p^2+p+1
  else:
    raise ValueError(f"Unexpected number of roots for p = {p}: {n_roots}")

@parallel(max(os.cpu_count()-1,1))  # leave 1 core free for other tasks
def N0(p):
  if not p.is_prime():
    raise ValueError(f"Input to N0 must be prime, got {p}")
  # let's try looking through divisors from the top down instead
  n = N_bound(p)
  primes = n.prime_divisors()
  # we know that it works for n, but we now repeatedly look at the
  # largest divisors of n to possibly replace n by a smaller number
  found = True
  while found:
    found = False
    for q in primes:
      if (p*R).reduce(power_mod(R(l[0]/l[1]), n//q, p) - 1) == 0 and (p*R).reduce(power_mod(R(l[1]/l[2]), n//q, p) - 1) == 0:
      # if trib(n//q, p) == 0 and trib(n//q - 1, p) == 0:
        found = True
        if n % q^2 != 0:
          primes.remove(q)  # no need to check this prime again
        n //= q
        break
  print(f"N0({p}) = {n}", end=", ")
  p21 = 1 if trib(n-1, p^2) == 0 else 0
  p22 = 1 if trib(n, p^2) == 0 else 0
  if p21 or p22:
    print(f"\np = {p}, n = {n}, {p21}{p22}")
  return (n,str(p21)+str(p22))

  # # print(p, end=' ')
  # for n in sorted(divisors(N_bound(p))):
  #   # print(f"Checking p = {p}, n = {n}...")
  #   tnp, tnp1 = trib(n, p), trib(n-1, p)
  #   if tnp == 0 and tnp1 == 0:
  #     print(f"N0({p}) = {n}", end=", ")
  #     p21 = 1 if trib(n-1, p^2) == 0 else 0
  #     p22 = 1 if trib(n, p^2) == 0 else 0
  #     if p21 or p22:
  #       print(f"\np = {p}, n = {n}, {p21}{p22}")
  #     return (n,str(p21)+str(p22))
  # else:
  #   raise ValueError(f"Could not find n such that trib(n) and trib(n-1) are divisible by p = {p}")

# trib(10, 17)

# [(p,trib(N_bound(p), p)) for p in primes(start=13,stop=100)]

# serial
# [(p,N0(p)) for p in primes(start=13,stop=150)]
# parallel
[(p[0][0],q) for p,q in N0([p for p in primes(start=13,stop=1000)])]

N0(13) = 168, N0(17) = 288, N0(19) = 360, N0(23) = 553, N0(29) = 840, N0(31) = 993, N0(37) = 1407, N0(41) = 1680, N0(43) = 1848, N0(47) = 46, N0(53) = 52, N0(59) = 3541, N0(61) = 3720, N0(67) = 4557, N0(71) = 5113, N0(73) = 5328, N0(79) = 6240, N0(83) = 6888, N0(89) = 8011, N0(97) = 9507, 
p = 97, n = 9507, 10
N0(101) = 10200, N0(103) = 102, 
p = 103, n = 102, 01
N0(107) = 11448, N0(109) = 11880, N0(113) = 12883, N0(127) = 16128, N0(131) = 17160, N0(137) = 18907, N0(139) = 19320, N0(149) = 22200, N0(151) = 22800, N0(157) = 24807, N0(163) = 162, N0(167) = 27888, N0(173) = 29928, N0(179) = 32221, N0(181) = 32943, N0(191) = 36673, N0(193) = 37248, N0(197) = 38808, N0(199) = 198, N0(211) = 44520, N0(223) = 49953, N0(257) = 256, N0(239) = 57120, N0(227) = 51528, N0(233) = 54288, N0(271) = 73440, N0(229) = 52671, N0(241) = 58080, N0(263) = 69168, N0(251) = 63253, N0(269) = 268, N0(277) = 76728, N0(281) = 78960, N0(283) = 80088, N0(293) = 85848, N0(311) = 310, N0(317) = 100807, N0(307) = 9424

[(13, (168, '00')),
 (17, (288, '00')),
 (19, (360, '00')),
 (23, (553, '00')),
 (29, (840, '00')),
 (31, (993, '00')),
 (37, (1407, '00')),
 (41, (1680, '00')),
 (43, (1848, '00')),
 (47, (46, '00')),
 (53, (52, '00')),
 (59, (3541, '00')),
 (61, (3720, '00')),
 (67, (4557, '00')),
 (71, (5113, '00')),
 (73, (5328, '00')),
 (79, (6240, '00')),
 (83, (6888, '00')),
 (89, (8011, '00')),
 (97, (9507, '10')),
 (101, (10200, '00')),
 (103, (102, '01')),
 (107, (11448, '00')),
 (109, (11880, '00')),
 (113, (12883, '00')),
 (127, (16128, '00')),
 (131, (17160, '00')),
 (137, (18907, '00')),
 (139, (19320, '00')),
 (149, (22200, '00')),
 (151, (22800, '00')),
 (157, (24807, '00')),
 (163, (162, '00')),
 (167, (27888, '00')),
 (173, (29928, '00')),
 (179, (32221, '00')),
 (181, (32943, '00')),
 (191, (36673, '00')),
 (193, (37248, '00')),
 (197, (38808, '00')),
 (199, (198, '00')),
 (211, (44520, '00')),
 (223, (49953, '00')),
 (257, (256, '00')),
 (239, (57120, '00')),
 (227, (51528, '00')),
